# Generative AI – Practical 5: Image Generation using Stable Diffusion
**Name:** YOUR NAME  |  **PRN:** YOUR PRN  |  **Batch:** YOUR BATCH  |  **Branch:** AIML

**Aim:** Implement a pretrained text-to-image Stable Diffusion pipeline, identify and use its components (text encoder, U-Net, scheduler, VAE), engineer prompts, and analyse how parameters affect output.

| Part | Content | Rubric criterion |
|---|---|---|
| 1 | Setup, pipeline, component inspection, **manual denoising loop** | Setup & Implementation |
| 2 | Prompt engineering, variations, negative prompts | Prompt Engineering |
| 3 | Quantitative analysis: steps, CFG, schedulers, prompts (CLIP score + sharpness) | Output & Parameter Analysis |
| 4 | Concept summary & viva preparation | Demonstration & Viva |

> Runtime → Change runtime type → **T4 GPU**, then *Run all*.

## Theory in brief
Stable Diffusion is a **latent diffusion model**. Instead of denoising 512×512×3 pixels, it works in a compressed **latent space** (4×64×64, i.e. 48× fewer values).

```
 prompt ──► [CLIP Text Encoder] ──► text embeddings (77×768) ─────────┐ (cross-attention)
                                                                      ▼
 random noise z_T (4×64×64) ──► [U-Net] predicts noise ε ──► [Scheduler] z_t → z_{t-1}  (repeat T steps)
                                                                      │
                                                       clean latent z_0
                                                                      ▼
                                                     [VAE Decoder] ──► 512×512 image
```
| Component | Role |
|---|---|
| **Tokenizer + CLIP text encoder** | Converts the prompt into 77 token embeddings (conditioning signal) |
| **VAE** (encoder/decoder) | Compresses image ↔ latent (factor 8 per side). Decoder turns the final latent into pixels |
| **U-Net** | Given noisy latent, timestep and text embedding, predicts the noise to remove; text enters via cross-attention |
| **Scheduler (sampler)** | Defines the noise schedule and the update rule that uses predicted noise to step z_t → z_{t-1} |
| **Classifier-free guidance** | Run U-Net with and without prompt: ε = ε_uncond + s·(ε_cond − ε_uncond), s = guidance scale |

# Part 1 – Setup & Implementation

### 1.1 Install & imports

In [ ]:
!pip install -q diffusers transformers accelerate safetensors pandas

In [ ]:
import torch, time, numpy as np, pandas as pd, matplotlib.pyplot as plt
from PIL import Image
from diffusers import (StableDiffusionPipeline, StableDiffusionImg2ImgPipeline, DPMSolverMultistepScheduler,
                       EulerDiscreteScheduler, EulerAncestralDiscreteScheduler, DDIMScheduler, PNDMScheduler)

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32
print("Device:", device, "| dtype:", dtype)
assert device == "cuda", "Enable GPU: Runtime > Change runtime type > T4 GPU"
if device == "cuda": print(torch.cuda.get_device_name(0))

### 1.2 Load the pretrained model

In [ ]:
MODEL_ID = "stable-diffusion-v1-5/stable-diffusion-v1-5"
pipe = StableDiffusionPipeline.from_pretrained(MODEL_ID, torch_dtype=dtype, safety_checker=None).to(device)
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)   # fast, high-quality default
pipe.enable_attention_slicing()
BASE_SCHED_CFG = pipe.scheduler.config
print(type(pipe).__name__)

### 1.3 Identify the major components

In [ ]:
def n_params(m): return sum(p.numel() for p in m.parameters())/1e6
comp = pd.DataFrame([
    ["Tokenizer",    type(pipe.tokenizer).__name__,    "-",                         "text -> 77 token ids"],
    ["Text encoder", type(pipe.text_encoder).__name__, f"{n_params(pipe.text_encoder):.0f}M", "token ids -> embeddings (77x768)"],
    ["U-Net",        type(pipe.unet).__name__,         f"{n_params(pipe.unet):.0f}M",         "predicts noise in latent space"],
    ["VAE",          type(pipe.vae).__name__,          f"{n_params(pipe.vae):.0f}M",          "latent <-> pixel space"],
    ["Scheduler",    type(pipe.scheduler).__name__,    "-",                         "denoising update rule / noise schedule"],
], columns=["Component", "Class", "Params", "Role"])
display(comp)
print("U-Net in/out channels:", pipe.unet.config.in_channels, "| VAE scaling factor:", pipe.vae.config.scaling_factor)
print("Latent size for 512x512 image: 4 x 64 x 64 =", 4*64*64, "values vs pixel space", 3*512*512)

### 1.4 Text-to-image using the high-level pipeline

In [ ]:
def gen(prompt, seed=42, **kw):
    g = torch.Generator(device).manual_seed(seed)
    return pipe(prompt, generator=g, **kw).images[0]

def grid(imgs, titles, cols=4, size=3.2, suptitle=None):
    rows = (len(imgs)+cols-1)//cols
    fig, ax = plt.subplots(rows, cols, figsize=(cols*size, rows*size*1.08))
    ax = np.atleast_1d(ax).flatten()
    for a in ax: a.axis("off")
    for a, im, t in zip(ax, imgs, titles): a.imshow(im); a.set_title(t, fontsize=9)
    if suptitle: fig.suptitle(suptitle, fontsize=12)
    plt.tight_layout(); plt.show()

prompt = "a cozy wooden cabin in a snowy pine forest at dusk, warm glowing windows, cinematic lighting, highly detailed"
img_pipe = gen(prompt, num_inference_steps=30, guidance_scale=7.5)
plt.imshow(img_pipe); plt.axis("off"); plt.title("High-level pipeline output"); plt.show()

### 1.5 Re-implementing the pipeline manually (tokenizer → text encoder → U-Net loop → scheduler → VAE)
This proves understanding of what `pipe(...)` does internally. We also capture intermediate latents to *see* denoising.

In [ ]:
@torch.no_grad()
def encode_text(text):
    ids = pipe.tokenizer(text, padding="max_length", max_length=pipe.tokenizer.model_max_length,
                         truncation=True, return_tensors="pt").input_ids.to(device)
    return pipe.text_encoder(ids)[0], ids

@torch.no_grad()
def decode_latents(z):
    x = pipe.vae.decode(z / pipe.vae.config.scaling_factor).sample          # latent -> pixels
    x = (x / 2 + 0.5).clamp(0, 1).permute(0, 2, 3, 1).float().cpu().numpy()[0]
    return Image.fromarray((x * 255).astype("uint8"))

@torch.no_grad()
def manual_generate(prompt, negative="", steps=30, cfg=7.5, seed=42, snapshots=()):
    cond, _   = encode_text(prompt)                                          # 1) text encoder
    uncond, _ = encode_text(negative)                                        #    empty / negative prompt
    emb = torch.cat([uncond, cond])
    pipe.scheduler.set_timesteps(steps, device=device)                       # 2) scheduler timesteps
    g = torch.Generator(device).manual_seed(seed)
    z = torch.randn((1, pipe.unet.config.in_channels, 64, 64), generator=g, device=device, dtype=dtype)
    z = z * pipe.scheduler.init_noise_sigma                                  # 3) start from pure noise
    snaps = {}
    for i, t in enumerate(pipe.scheduler.timesteps):                         # 4) denoising loop
        zin = pipe.scheduler.scale_model_input(torch.cat([z, z]), t)
        eps = pipe.unet(zin, t, encoder_hidden_states=emb).sample            #    U-Net predicts noise
        e_u, e_c = eps.chunk(2)
        eps = e_u + cfg * (e_c - e_u)                                        #    classifier-free guidance
        z = pipe.scheduler.step(eps, t, z).prev_sample                       #    scheduler update z_t -> z_{t-1}
        if i in snapshots: snaps[i] = decode_latents(z)
    return decode_latents(z), snaps                                          # 5) VAE decoder

STEPS = 30
snap_idx = [0, 3, 8, 15, 22, STEPS-1]
img_manual, snaps = manual_generate(prompt, steps=STEPS, cfg=7.5, seed=42, snapshots=snap_idx)
grid([snaps[i] for i in snap_idx] + [img_manual],
     [f"after step {i+1}/{STEPS}" for i in snap_idx] + ["FINAL"], cols=4, suptitle="Denoising progress (latent decoded by VAE at each stage)")

In [ ]:
# Verify: manual implementation == high-level pipeline (same seed, steps, scheduler)
a = np.array(gen(prompt, num_inference_steps=STEPS, guidance_scale=7.5)).astype(float)
b = np.array(img_manual).astype(float)
print(f"Mean abs pixel difference pipeline vs manual: {np.abs(a-b).mean():.3f}  (≈0 means identical up to fp16 noise)")
grid([Image.fromarray(a.astype('uint8')), img_manual], ["pipe(...)", "manual loop"], cols=2, size=5)

### 1.6 Visualising the latent space (what the U-Net actually denoises)

In [ ]:
g = torch.Generator(device).manual_seed(42)
z0 = torch.randn((1,4,64,64), generator=g, device=device, dtype=dtype)
print("Latent tensor shape:", tuple(z0.shape))
fig, ax = plt.subplots(1, 5, figsize=(15, 3))
for k in range(4): ax[k].imshow(z0[0,k].float().cpu(), cmap="viridis"); ax[k].set_title(f"noise latent ch {k}"); ax[k].axis("off")
ax[4].imshow(img_manual); ax[4].set_title("decoded final image"); ax[4].axis("off"); plt.show()
# VAE round trip: image -> latent -> image
with torch.no_grad():
    x = torch.tensor(np.array(img_manual.resize((512,512)))/127.5-1, dtype=dtype, device=device).permute(2,0,1)[None]
    lat = pipe.vae.encode(x).latent_dist.sample() * pipe.vae.config.scaling_factor
print("Image", tuple(x.shape), "-> latent", tuple(lat.shape), f"(compression x{x.numel()/lat.numel():.0f})")
grid([img_manual, decode_latents(lat)], ["original", "after VAE encode+decode"], cols=2, size=4)

# Part 2 – Prompt Engineering & Image Generation

**Prompt structure used:** `[subject] + [action/setting] + [style/medium] + [lighting] + [camera/composition] + [quality tags]`

Negative prompts tell the model what to steer *away from* (they replace the empty "unconditional" prompt in classifier-free guidance).

### 2.1 Vague vs engineered prompts

In [ ]:
prompts_detail = {
 "L1 vague":        "a dog",
 "L2 + setting":    "a golden retriever running on a beach",
 "L3 + style":      "a golden retriever running on a beach, digital painting",
 "L4 + lighting":   "a golden retriever running on a beach at golden hour, digital painting, warm soft lighting",
 "L5 engineered":   "a golden retriever running along a beach at golden hour, splashing water, sharp focus, 85mm lens, shallow depth of field, warm soft lighting, ultra detailed, professional photography",
}
imgs = [gen(p, num_inference_steps=30) for p in prompts_detail.values()]
grid(imgs, list(prompts_detail.keys()), cols=5, size=3.4, suptitle="Increasing prompt specificity (same seed)")

### 2.2 Style / modifier variations (controlled change of one attribute)

In [ ]:
subject = "an old lighthouse on a rocky coast"
styles = ["oil painting, impressionist", "photorealistic, 35mm film", "anime style, studio ghibli inspired",
          "pencil sketch, black and white", "low-poly 3D render", "watercolor, soft pastel colors"]
imgs = [gen(f"{subject}, {s}, highly detailed", num_inference_steps=30, seed=11) for s in styles]
grid(imgs, styles, cols=3, size=4.2, suptitle=f"Style variation – subject: {subject}")

### 2.3 Lighting / mood variations

In [ ]:
moods = ["sunrise, golden light", "stormy night, dramatic lightning", "foggy morning, muted tones", "neon cyberpunk lighting"]
imgs = [gen(f"{subject}, {m}, cinematic", num_inference_steps=30, seed=11) for m in moods]
grid(imgs, moods, cols=4, size=3.4, suptitle="Mood / lighting variation")

### 2.4 Negative prompts

In [ ]:
p = "close-up portrait photo of an elderly fisherman, detailed face, natural light, 85mm"
neg = "blurry, low quality, deformed, extra fingers, bad anatomy, watermark, text, oversaturated, cartoon"
cases = {"no negative": "", "generic negative": neg, "negative: 'beard'": "beard"}
imgs = [gen(p, negative_prompt=n, num_inference_steps=30, seed=5) for n in cases.values()]
grid(imgs, list(cases.keys()), cols=3, size=4.2, suptitle="Effect of negative prompts (same seed)")

### 2.5 Prompt-token weighting by repetition / emphasis & token limit

In [ ]:
long_p = "a castle " + "on a hill, misty, " * 30 + "with a red dragon"
ids = pipe.tokenizer(long_p).input_ids
print(f"Tokens: {len(ids)}  | limit: {pipe.tokenizer.model_max_length} -> tokens beyond the limit are truncated (the 'red dragon' is lost)")
imgs = [gen("a castle on a hill, red dragon flying above", seed=3, num_inference_steps=30),
        gen("a castle on a hill, red dragon, red dragon, red dragon, huge dragon dominating the sky", seed=3, num_inference_steps=30),
        gen(long_p, seed=3, num_inference_steps=30)]
grid(imgs, ["normal", "repeated emphasis", "over-long (truncated)"], cols=3, size=4.2)

### 2.6 Same prompt, different seeds (diversity) and same seed (reproducibility)

In [ ]:
imgs = [gen(prompt, seed=s, num_inference_steps=25) for s in [1, 2, 3, 4]]
grid(imgs, [f"seed {s}" for s in [1,2,3,4]], cols=4, suptitle="Different seeds -> different starting noise -> different compositions")
x1, x2 = np.array(gen(prompt, seed=7, num_inference_steps=25)), np.array(gen(prompt, seed=7, num_inference_steps=25))
print("Same seed gives identical image:", np.array_equal(x1, x2))

# Part 3 – Output & Parameter Analysis

To compare outputs **objectively** we use two metrics:
* **CLIP score** – cosine similarity between CLIP image embedding and prompt embedding ×100 (higher = image matches prompt better).
* **Sharpness / detail** – variance of the Laplacian of the grayscale image (higher = more high-frequency detail).
* **Latency** – seconds per image.

Averaged over several prompts and seeds to reduce noise.

In [ ]:
from transformers import CLIPModel, CLIPProcessor
clip = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device).eval()
clip_proc = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

@torch.no_grad()
def clip_score(image, text):
    inp = clip_proc(text=[text], images=image, return_tensors="pt", padding=True, truncation=True).to(device)
    out = clip(**inp)
    return 100 * torch.cosine_similarity(out.image_embeds, out.text_embeds).item()

def sharpness(image):
    g = np.array(image.convert("L")).astype(float)
    lap = 4*g[1:-1,1:-1] - g[:-2,1:-1] - g[2:,1:-1] - g[1:-1,:-2] - g[1:-1,2:]
    return lap.var()

EVAL_PROMPTS = [
  "a red vintage car parked on a rainy city street at night, neon reflections",
  "a bowl of fresh fruit on a wooden table, soft window light, photorealistic",
  "an astronaut riding a horse on mars, digital art",
]
SEEDS = [0, 1]

def evaluate(**gen_kwargs):
    scores, sharps, times, sample = [], [], [], None
    for p in EVAL_PROMPTS:
        for s in SEEDS:
            t = time.time(); im = gen(p, seed=s, **gen_kwargs); times.append(time.time()-t)
            scores.append(clip_score(im, p)); sharps.append(sharpness(im))
            if sample is None: sample = im
    return dict(clip=np.mean(scores), sharp=np.mean(sharps), time=np.mean(times), img=sample)

def table_and_plots(results, xlabel):
    df = pd.DataFrame([{xlabel: k, "CLIP score": v["clip"], "Sharpness": v["sharp"], "Time (s)": v["time"]} for k, v in results.items()])
    display(df.round(2))
    fig, ax = plt.subplots(1, 3, figsize=(14, 3.2))
    for a, col in zip(ax, ["CLIP score", "Sharpness", "Time (s)"]):
        a.plot(df[xlabel].astype(str), df[col], marker="o"); a.set_title(col); a.set_xlabel(xlabel); a.grid(alpha=.3)
    plt.tight_layout(); plt.show()
    return df

### 3.1 Inference steps

In [ ]:
pipe.scheduler = DPMSolverMultistepScheduler.from_config(BASE_SCHED_CFG)
res_steps = {s: evaluate(num_inference_steps=s, guidance_scale=7.5) for s in [5, 10, 20, 30, 50]}
df_steps = table_and_plots(res_steps, "steps")
grid([v["img"] for v in res_steps.values()], [f"{k} steps" for k in res_steps], cols=5, size=3.4)

### 3.2 Guidance scale (CFG)

In [ ]:
res_cfg = {c: evaluate(num_inference_steps=25, guidance_scale=c) for c in [1, 3, 5, 7.5, 10, 15, 20]}
df_cfg = table_and_plots(res_cfg, "guidance")
grid([v["img"] for v in res_cfg.values()], [f"CFG {k}" for k in res_cfg], cols=7, size=2.6)

### 3.3 Scheduler / sampler comparison

In [ ]:
schedulers = {"PNDM": PNDMScheduler, "DDIM": DDIMScheduler, "Euler": EulerDiscreteScheduler,
              "Euler-A": EulerAncestralDiscreteScheduler, "DPM++ 2M": DPMSolverMultistepScheduler}
res_sched = {}
for name, cls in schedulers.items():
    pipe.scheduler = cls.from_config(BASE_SCHED_CFG)
    res_sched[name] = evaluate(num_inference_steps=20, guidance_scale=7.5)
df_sched = table_and_plots(res_sched, "scheduler")
grid([v["img"] for v in res_sched.values()], list(res_sched), cols=5, size=3.4)
pipe.scheduler = DPMSolverMultistepScheduler.from_config(BASE_SCHED_CFG)

### 3.4 Scheduler × steps (convergence speed)

In [ ]:
rows = []
for name in ["DDIM", "Euler", "Euler-A", "DPM++ 2M"]:
    pipe.scheduler = schedulers[name].from_config(BASE_SCHED_CFG)
    for st in [5, 10, 20, 40]:
        r = evaluate(num_inference_steps=st, guidance_scale=7.5)
        rows.append({"scheduler": name, "steps": st, "CLIP": r["clip"]})
pipe.scheduler = DPMSolverMultistepScheduler.from_config(BASE_SCHED_CFG)
df_conv = pd.DataFrame(rows)
display(df_conv.pivot(index="steps", columns="scheduler", values="CLIP").round(2))
for name, d in df_conv.groupby("scheduler"): plt.plot(d["steps"], d["CLIP"], marker="o", label=name)
plt.xlabel("steps"); plt.ylabel("CLIP score"); plt.legend(); plt.title("Convergence of schedulers"); plt.grid(alpha=.3); plt.show()

### 3.5 Prompt variation analysis (specificity vs prompt alignment)

In [ ]:
rows = []
for lvl, p in prompts_detail.items():
    im = gen(p, num_inference_steps=30, seed=42)
    rows.append({"prompt level": lvl, "CLIP vs own prompt": clip_score(im, p), "Sharpness": sharpness(im)})
df_prompt = pd.DataFrame(rows); display(df_prompt.round(2))
# negative prompt effect on CLIP + sharpness
rows = []
for name, n in cases.items():
    im = gen("close-up portrait photo of an elderly fisherman, detailed face, natural light, 85mm", negative_prompt=n, num_inference_steps=30, seed=5)
    rows.append({"negative": name, "Sharpness": sharpness(im)})
display(pd.DataFrame(rows).round(2))

### 3.6 Summary table (auto-generated)

In [ ]:
best_steps = df_steps.loc[df_steps["CLIP score"].idxmax(), "steps"]
best_cfg   = df_cfg.loc[df_cfg["CLIP score"].idxmax(), "guidance"]
fast       = df_sched.loc[df_sched["Time (s)"].idxmin(), "scheduler"]
best_sched = df_sched.loc[df_sched["CLIP score"].idxmax(), "scheduler"]
print(f"Best CLIP at steps = {best_steps}; best CLIP at CFG = {best_cfg}")
print(f"Fastest scheduler = {fast}; highest CLIP scheduler = {best_sched}")
print("Time per step approx (s):", round(np.polyfit(df_steps['steps'], df_steps['Time (s)'], 1)[0], 3))

### 3.7 Written analysis  *(verify against YOUR tables/plots above and edit numbers)*
**Inference steps.** Image quality and CLIP score rise rapidly from 5 → 20 steps (coarse blobs → recognisable structure → fine detail) and then plateau; beyond ~30 steps extra steps cost time (latency grows linearly with steps because each step is one U-Net forward pass ×2 for CFG) with negligible gain.

**Guidance scale.** CFG ≈ 1 mostly ignores the prompt (blurry/generic, low CLIP). CLIP score rises up to ≈ 7–12 as the model follows the prompt more strictly. At ≥ 15–20 images become over-saturated/high-contrast with artefacts and reduced diversity, and sharpness rises artificially. Optimum trade-off is ≈ 7–10.

**Schedulers.** All samplers solve the same reverse-diffusion process with different numerical solvers. DPM++ 2M / Euler converge in far fewer steps (≈ 15–25) than DDIM/PNDM at 5–10 steps. Ancestral samplers (Euler-A) inject fresh noise every step so results don't converge to a single image as steps increase. Per-step cost is nearly equal, so the better solver is simply the one needing fewer steps.

**Prompt variations.** Specific prompts (subject + style + lighting + camera) increase control and prompt alignment compared with vague prompts, which fall back to the dataset's average appearance. Style and mood words change global appearance while the composition stays similar for the same seed, because the seed fixes the initial latent noise and the layout is decided in the first high-noise steps.

**Negative prompts.** They push the denoising direction away from unwanted concepts (blur, extra fingers, watermark), improving cleanliness; a targeted negative (e.g. 'beard') directly removes the concept.

**Limitations.** CLIP score is a coarse proxy of alignment (not aesthetics); only 3 prompts × 2 seeds were averaged; SD-1.5 is trained at 512×512 and has a 77-token limit.

# Part 4 – Demonstration & Viva Preparation

### 4.1 One-cell end-to-end demo (use this during your demonstration)

In [ ]:
def demo(prompt, negative="blurry, low quality, deformed, watermark", steps=25, cfg=7.5, seed=42, scheduler="DPM++ 2M"):
    pipe.scheduler = schedulers[scheduler].from_config(BASE_SCHED_CFG)
    t = time.time(); im = gen(prompt, negative_prompt=negative, num_inference_steps=steps, guidance_scale=cfg, seed=seed)
    plt.figure(figsize=(5,5)); plt.imshow(im); plt.axis("off")
    plt.title(f"{scheduler} | steps={steps} | CFG={cfg} | seed={seed} | {time.time()-t:.1f}s | CLIP={clip_score(im, prompt):.1f}", fontsize=8); plt.show()
    pipe.scheduler = DPMSolverMultistepScheduler.from_config(BASE_SCHED_CFG)
    return im

_ = demo("a futuristic city floating in the clouds at sunset, ultra detailed, concept art")

### 4.2 Bonus: image-to-image (conditioning on an existing image)

In [ ]:
img2img = StableDiffusionImg2ImgPipeline(**pipe.components).to(device)
init = gen("a simple pencil sketch of a mountain lake with a small boat", num_inference_steps=25).resize((512,512))
outs, tt = [init], ["input"]
for s in [0.3, 0.55, 0.8, 1.0]:
    g = torch.Generator(device).manual_seed(1)
    outs.append(img2img("a vibrant oil painting of a mountain lake at sunrise", image=init, strength=s, guidance_scale=7.5,
                        generator=g, num_inference_steps=30).images[0]); tt.append(f"strength {s}")
grid(outs, tt, cols=5, size=3.4, suptitle="strength = fraction of noise added to the input before denoising")

## Conclusion
Implemented a pretrained Stable Diffusion v1.5 text-to-image pipeline, re-implemented its internals manually (tokenizer → CLIP text encoder → U-Net denoising loop with classifier-free guidance → scheduler → VAE decoder) and verified it against the library pipeline. Prompt engineering, negative prompts, seeds, steps, CFG and schedulers were analysed both visually and with CLIP-score / sharpness / latency metrics.

---
# 🎓 Viva Cheat-Sheet

**Q1. What is a diffusion model?** A generative model trained to reverse a gradual noising process. Forward: add Gaussian noise over T steps until the image is pure noise. Reverse: a network learns to predict the noise at each step so we can start from noise and denoise into an image.

**Q2. What is denoising?** At each step the U-Net predicts the noise ε in z_t; the scheduler subtracts a scaled amount of it to get z_{t-1}. Early steps decide global layout; late steps add fine texture.

**Q3. Why *latent* diffusion? What is the latent representation?** Pixel-space diffusion is expensive. The VAE encoder compresses 512×512×3 → 64×64×4 (48× smaller), the diffusion runs there, and the VAE decoder reconstructs pixels. Much faster/cheaper with little quality loss.

**Q4. How does the prompt condition the image?** CLIP text encoder → 77×768 embeddings → injected in the U-Net through **cross-attention** layers (image features = queries, text embeddings = keys/values) at every denoising step.

**Q5. What is classifier-free guidance / guidance scale?** Run the U-Net with the prompt and with an empty (or negative) prompt: ε = ε_u + s(ε_c − ε_u). Larger s → stronger prompt adherence but less diversity and possible artefacts.

**Q6. What does a negative prompt do?** It replaces the empty prompt in the unconditional branch, so the guidance direction moves *away* from those concepts.

**Q7. What does the scheduler do? Why do samplers differ?** It defines the noise schedule and the numerical update from z_t to z_{t-1}. Different ODE/SDE solvers (DDIM, Euler, DPM++) need different numbers of steps for the same quality; ancestral ones add noise each step and are stochastic.

**Q8. Why does the seed matter?** It fixes the initial random latent; same seed + same settings = same image. Different seeds give different layouts.

**Q9. Why does more steps ≠ always better?** Quality saturates (solver error is already small); cost grows linearly with steps.

**Q10. What does `strength` do in img2img?** Determines how far into the noise schedule we start: image is encoded to a latent, noise added for strength×T steps, then denoised with the prompt. Low strength = stays near input; high = nearly ignores it.

**Q11. What is the role of the U-Net's timestep embedding?** Tells the network how noisy the input is so it predicts the right amount of noise.

**Q12. Why 77 tokens?** CLIP text encoder's context length; longer prompts are truncated.

**Q13. Limitations?** Hands/text/counting errors, 512px training resolution, biases from LAION data, CLIP-score is only a proxy metric.

**Q14. fp16 – why?** Halves memory and speeds inference on GPU with negligible quality loss.

**Q15. How is Stable Diffusion trained?** Take latent z_0, sample t and noise ε, form z_t, train the U-Net to predict ε given (z_t, t, text) with MSE loss.